# Sequence-to-sequence RNN for Web Traffic Time Series Forecasting

Forecast 62 days of daily pageviews (2017-09-13 to 2017-11-13) for 145,063
Wikipedia articles. Evaluation metric: SMAPE.

## Results

| Evaluation | Period | Baseline (49-day median) | This model |
|---|---|---|---|
| Held-out backtest | 2017-07-11 to 2017-09-10 | 42.773 | 42.633 |
| Kaggle leaderboard | 2017-09-13 to 2017-11-13 | not submitted | **37.41** |

The backtest difference of 0.140 SMAPE has a 95% bootstrap confidence interval
of [-1.154, +0.968] and is therefore not distinguishable from zero. The two
rows are measured over different periods and are not comparable to each other;
the leaderboard figure should not be read as an improvement over 42.773.

Establishing whether the model beats the median baseline *on the leaderboard*
requires submitting the baseline as well. Until then the honest summary is that
the model matches the baseline on held-out data and its leaderboard standing
relative to that baseline is unmeasured.

## Architecture

Encoder-decoder GRU with additive attention over encoder states, following the
general structure of the competition's winning entry (Arthur Suilin, 2017),
reimplemented in PyTorch. 157,665 parameters.

## Design decisions

Each choice is recorded against a measured property of `train_2.csv` rather
than convention.

| Decision | Alternative rejected | Justification |
|---|---|---|
| `log1p` transform | raw counts | Per-series medians span 0 to 19.4M views/day. Untransformed, the objective is dominated by a few articles: `Main_Page` alone averages 16.3M views. |
| Per-series normalisation | one global normalisation | Scale varies across seven orders of magnitude, so no single mean and variance describes the population. |
| L1 loss on normalised `log1p` | MSE | SMAPE is minimised by the conditional median; MSE fits the conditional mean. The mean exceeds the median for 91% of series (p90 ratio 1.62), so MSE imposes a systematic upward bias. AutoARIMA, fit by maximum likelihood, exhibited precisely this failure: forecast/actual ratio 1.257, 61.41 SMAPE against the baseline's 42.50. |
| Explicit observed-mask channel | zero-filling or interpolation | 6.03% of cells are missing. 14.3% of series have leading gaps of median length 210 days, because the article did not yet exist. Zero-filling asserts traffic was genuinely zero; only 1.41% of observed values are zero. Missing-ness is itself informative about page age. |
| Autoregressive decoder, prediction detached | direct multi-horizon head | Consuming its own previous output teaches the model conservatism over a 62-step horizon. Detaching prevents backpropagation through the full chain, which is expensive and unstable. |
| Random forecast origins per epoch | fixed training windows | Functions as data augmentation: with roughly 500 admissible origins per series across 145,063 series, the model sees a effectively non-repeating stream of windows. |
| Round and clip at zero | continuous output | SMAPE is 0 when actual and forecast are both zero, but returns the maximum penalty of 200 for any positive forecast against a zero actual. Rounding converts small positive predictions to exact zeros. |
| Metadata as learned embeddings | one-hot encoding | Project, access and agent have 9, 3 and 2 levels. Embeddings let related levels share structure; the original used one-hot, and the distinction is not expected to be material. |

## Divergences from the reference solution

Recorded so the gap is explicit rather than implied.

**Attention formulation.** This notebook uses classical additive (Bahdanau)
attention, recomputed at every decoder step. Suilin rejected that approach on
cost grounds — it must be recalculated over the full history at each of 62
steps — and instead read the encoder at fixed lag points (365, 182, 91 days)
smoothed by a learned convolution, applied once and shared across steps. He
reports that his best public score used no attention at all, relying only on
lagged datapoints. With a 180-day encoder the recomputation cost here is
tolerable, but the mechanism is not the one that won.

**Absent features.** Year-to-year and quarter-to-quarter autocorrelation, page
popularity, and explicit lagged pageviews are not provided. The 180-day
lookback cannot reach a 365-day lag, so no annual structure is available to the
model by any route.

**Absent variance reduction.** Suilin averaged 30 checkpoints across 3 seeds
with ASGD weight averaging, and identifies this as the mechanism by which his
leaderboard score matched his validation score. This is a single run on a
single seed, and carries the corresponding variance.

**Other.** COCOB optimiser (Adam with one-cycle used instead), SMAC3
hyperparameter search, and RNN activation regularisation are not implemented.

## Execution

Requires a GPU runtime. Run the login cell alone and wait for confirmation
before continuing: `kagglehub.login()` renders a widget and returns
immediately, so a download issued in the same cell executes before
authentication completes and fails with HTTP 401.

In [1]:
import importlib.util, os, subprocess, sys

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/input")
print("environment:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")

for pkg in ["kagglehub"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
elif IN_COLAB:
    print("\n*** NO GPU ***  Runtime > Change runtime type > GPU, then Run all again.")

environment: Colab
torch 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4 | 16 GB


## 1. Authentication

`kagglehub.login()` displays an input widget and returns control immediately;
it does not block until a token is submitted. The download is therefore placed
in a separate cell — issuing both from one cell sends an unauthenticated
request and fails with HTTP 401.

Run this cell, submit the API token, and wait for `Kaggle credentials set.`
before continuing. The competition rules must also have been accepted by the
account owning the token.

In [5]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


## 2. Data acquisition

`kagglehub` caches the archive after first download, so re-execution is
inexpensive. The directory walk locates the extraction path explicitly, as the
archive layout is not guaranteed to be flat.

In [6]:
import os
try:
    import kagglehub
except ImportError:
    !pip -q install kagglehub
    import kagglehub

root = kagglehub.competition_download("web-traffic-time-series-forecasting")
print("Data source import complete. downloaded to:", root)

# Archive layout is not guaranteed to be flat; locate the extraction directory.
DATA_DIR = None
for d, _, files in os.walk(root):
    if "train_2.csv.zip" in files:
        DATA_DIR = d
        break
assert DATA_DIR, "train_2.csv.zip not found under " + root
print("DATA_DIR:", DATA_DIR)
print("contents:", os.listdir(DATA_DIR))

100%|██████████| 583M/583M [00:06<00:00, 90.0MB/s]

Extracting files...


Data source import complete. downloaded to: /root/.cache/kagglehub/competitions/web-traffic-time-series-forecasting
DATA_DIR: /root/.cache/kagglehub/competitions/web-traffic-time-series-forecasting
contents: ['train_1.csv.zip', 'key_2.csv.zip', 'sample_submission_2.csv.zip', 'train_2.csv.zip', 'sample_submission_1.csv.zip', 'key_1.csv.zip']


In [7]:
import time, warnings, gc
import numpy as np, pandas as pd
import torch.nn as nn
warnings.filterwarnings("ignore")

H, GAP, SEED = 62, 2, 0
LOOKBACK = 180          # encoder window, in days. Too short to reach a 365-day
                        # lag, so annual structure is unavailable to this model.
HIDDEN   = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_GPU = DEVICE.type == "cuda"

# Budget scales with hardware. The CPU configuration verifies that the pipeline
# executes; it is not sufficient to train a usable model.
SAMPLES = 400_000 if ON_GPU else 40_000
EPOCHS  = 12      if ON_GPU else 2
BATCH   = 512     if ON_GPU else 128

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"device={DEVICE}  samples/epoch={SAMPLES:,}  epochs={EPOCHS}  batch={BATCH}")

device=cuda  samples/epoch=400,000  epochs=12  batch=512


In [8]:
# Parse numeric columns directly as float32. The frame is 145,063 x 803; float64
# would occupy roughly 930 MB where 466 MB suffices.
csv = os.path.join(DATA_DIR, "train_2.csv.zip")
hdr = pd.read_csv(csv, nrows=0)
dtypes = {c: "float32" for c in hdr.columns if c != "Page"}
train = pd.read_csv(csv, dtype=dtypes)

cols  = train.columns[1:]
pages = train["Page"].to_numpy()
V     = train[cols].to_numpy(dtype="float32")          # (145063, 803)
dates = pd.to_datetime(cols)
dow   = np.array([d.dayofweek for d in dates], dtype="int64")

parts = pd.Series(pages).str.rsplit("_", n=3, expand=True)
parts.columns = ["name", "project", "access", "agent"]
codes = np.stack([pd.Categorical(parts[c]).codes
                  for c in ["project", "access", "agent"]], axis=1).astype("int64")

del train, parts, hdr
gc.collect()

med     = np.nanmedian(V, axis=1)
TRN_END = V.shape[1] - (H + GAP)
print(f"{V.shape[0]:,} series x {V.shape[1]} days ({V.nbytes/1e6:.0f} MB)")
print(f"fit through {dates[TRN_END-1].date()} | "
      f"backtest {dates[-H].date()}..{dates[-1].date()}")

145,063 series x 803 days (466 MB)
fit through 2017-07-08 | backtest 2017-07-11..2017-09-10


## 4. Training sample construction

A training example is one `(series, forecast origin)` pair. The encoder
receives `LOOKBACK` days ending at the origin; the target is the following 62
days, matching the competition horizon.

Origins are resampled every epoch rather than fixed, which acts as data
augmentation and limits memorisation of any particular date range.

Normalisation is per-example, computed over the observed values in the encoder
window only, and inverted at prediction time. Statistics are computed from
observed values alone so that missing days do not bias the mean downward.

The standard-deviation floor of 0.5 is a correctness requirement rather than a
tuning parameter. Series with flat traffic have near-zero standard deviation
over the window, and dividing by it yields normalised targets orders of
magnitude too large. With a floor of 1e-3 the training loss stalled at 4.54 in
normalised units where values near 1 are expected; raising the floor to 0.5 and
clipping normalised targets to +/-10 reduced it to 0.82. The failure is silent
— the model trains and converges to nothing useful without raising an error.

Three static features accompany each window: the series mean and standard
deviation in log space, which restore the scale information that per-series
normalisation removes, and the observed fraction of the encoder window, which
proxies page age.

In [9]:
def make_sample(i, o, L=LOOKBACK, Varr=None, dowarr=None):
    """Encoder window ends at o (exclusive); target is the next H days."""
    Varr   = V   if Varr   is None else Varr
    dowarr = dow if dowarr is None else dowarr
    enc, tgt = Varr[i, o - L:o], Varr[i, o:o + H]
    m_enc, m_tgt = ~np.isnan(enc), ~np.isnan(tgt)
    x = np.log1p(np.nan_to_num(enc, nan=0.0))
    y = np.log1p(np.nan_to_num(tgt, nan=0.0))

    obs = x[m_enc]
    mu  = obs.mean() if obs.size else 0.0
    # Required floor, not a tuned value. Flat series have near-zero standard
    # deviation over the window; dividing by it produces normalised targets orders
    # of magnitude too large. The failure is silent - training proceeds and
    # converges to nothing. Observed loss 4.54 at a 1e-3 floor, 0.82 at 0.5.
    sd  = max(float(obs.std() if obs.size else 1.0), 0.5)

    enc_feat = np.stack([(x - mu) / sd * m_enc, m_enc.astype("float32")], axis=1)
    age  = min(int(m_enc.sum()), L) / L
    stat = np.array([mu / 10.0, sd, age], dtype="float32")
    yn   = np.clip((y - mu) / sd, -10.0, 10.0)

    return (torch.from_numpy(enc_feat.astype("float32")),
            torch.from_numpy(dowarr[o:o + H]),
            torch.from_numpy(codes[i]),
            torch.from_numpy(stat),
            torch.tensor([mu, sd], dtype=torch.float32),
            torch.from_numpy(yn.astype("float32")),
            torch.from_numpy(m_tgt.astype("float32")))


class Windows(torch.utils.data.Dataset):
    def __init__(self, n_samples, end, seed=SEED):
        rng = np.random.default_rng(seed)
        self.series = rng.integers(0, V.shape[0], size=n_samples)
        self.origin = rng.integers(LOOKBACK, end - H, size=n_samples)

    def __len__(self):
        return len(self.series)

    def __getitem__(self, k):
        return make_sample(int(self.series[k]), int(self.origin[k]))

## 5. Model

### Encoder and decoder

A single-layer GRU encodes the lookback window. The decoder is a `GRUCell`
advanced one step per forecast day, consuming at each step its own previous
prediction, a day-of-week embedding, the static page features, and an attention
context vector.

The previous prediction is passed through `.detach()`. This retains the
conservatism that autoregressive feedback produces — error accumulates across
62 steps, so an extreme early prediction degrades the entire trajectory — while
avoiding backpropagation through the full chain.

### Attention

Additive (Bahdanau) attention: at each decoder step the current hidden state
scores every encoder position, and the softmax-weighted sum forms the context
vector.

This is a deliberate divergence from the reference solution, and the trade-off
should be understood. Suilin rejected this formulation because it must be
recomputed from scratch at every prediction step across all historical
datapoints, which is prohibitive for series approaching 800 days. His
alternative read the encoder at fixed lag points — 365, 182 and 91 days before
each prediction day — smoothed by a learned convolution applied once and shared
across all steps.

With a 180-day encoder the recomputation cost here is acceptable. The
substantive loss is reach: a 180-day window cannot see a year back, so annual
structure is unavailable to this model regardless of the attention mechanism.

### Loss

Masked L1 on normalised log values. SMAPE cannot be optimised directly — it is
a step function when the true value is zero and undefined when both values are
zero. Suilin identifies MAE on `log1p` as an acceptable substitute, "close
enough to SMAPE for training purposes"; the alternative is his smoothed
differentiable SMAPE variant.

The mask excludes missing target days from the loss entirely, so the model is
never trained toward an imputed value.

In [10]:
class Seq2Seq(nn.Module):
    def __init__(self, hidden=HIDDEN, emb=(9, 3, 2), emb_dim=(6, 3, 2)):
        super().__init__()
        self.encoder = nn.GRU(2, hidden, batch_first=True)
        self.embs    = nn.ModuleList([nn.Embedding(n, d) for n, d in zip(emb, emb_dim)])
        self.dow_emb = nn.Embedding(7, 4)
        # Decoder input: previous prediction, day-of-week embedding, static page
        # features (mu, sd, observed fraction plus metadata), attention context.
        dec_in = 1 + 4 + (sum(emb_dim) + 3) + hidden
        self.decoder = nn.GRUCell(dec_in, hidden)
        self.attn    = nn.Linear(hidden * 2, 1)
        self.head    = nn.Linear(hidden * 2, 1)

    def forward(self, enc_feat, dow_b, codes_b, stat):
        enc_out, h = self.encoder(enc_feat)
        h = h[0]
        static = torch.cat([e(codes_b[:, k]) for k, e in enumerate(self.embs)] + [stat], 1)
        prev, outs = enc_feat[:, -1, 0:1], []
        for t in range(H):
            # Additive (Bahdanau) attention, recomputed over every encoder position
            # at each of the 62 steps. Suilin rejected this cost for ~800-day series
            # and read fixed lag points instead; a 180-day encoder makes it viable.
            q   = h.unsqueeze(1).expand(-1, enc_out.size(1), -1)
            w   = torch.softmax(self.attn(torch.cat([enc_out, q], 2)).squeeze(2), dim=1)
            ctx = torch.bmm(w.unsqueeze(1), enc_out).squeeze(1)
            h   = self.decoder(torch.cat([prev, self.dow_emb(dow_b[:, t]), static, ctx], 1), h)
            y   = self.head(torch.cat([h, ctx], 1))
            outs.append(y)
            prev = y.detach()
        return torch.cat(outs, dim=1)


def masked_l1(pred, target, mask):
    return ((pred - target).abs() * mask).sum() / mask.sum().clamp(min=1.0)


model = Seq2Seq().to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

157,665 parameters


## 6. Training

400,000 windows across 12 epochs at batch size 512 gives 9,372 optimiser steps,
comparable to the 10,500-11,500 steps reported for the reference solution.
Training loss declined monotonically from 0.749 to 0.639 and had not plateaued
at the end of the schedule, indicating the budget rather than the architecture
was the binding constraint.

Gradient clipping at norm 1.0 is retained from the original. Adam with a
one-cycle schedule replaces COCOB; the latter removes learning-rate tuning,
which is a convenience rather than an accuracy mechanism.

No early stopping is used. Suilin reports that validation performance correlates
only weakly with future performance, making stop-step selection unreliable; his
solution instead averaged checkpoints across a plausible region. That averaging
is not implemented here, so this run retains the seed variance those measures
were designed to remove.

In [11]:
ds = Windows(SAMPLES, TRN_END)
dl = torch.utils.data.DataLoader(
    ds, batch_size=BATCH, shuffle=True,
    num_workers=2 if (IN_COLAB or IN_KAGGLE) else 0,
    pin_memory=ON_GPU, drop_last=True)

opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3,
                                            total_steps=EPOCHS * len(dl))
print(f"{len(dl)} steps/epoch x {EPOCHS} epochs = {len(dl)*EPOCHS:,} steps")

for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = nb = 0
    for enc, d_, c_, s_, _, y, m in dl:
        enc, d_, c_, s_, y, m = [t.to(DEVICE, non_blocking=True)
                                 for t in (enc, d_, c_, s_, y, m)]
        loss = masked_l1(model(enc, d_, c_, s_), y, m)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        tot += loss.item(); nb += 1
    print(f"epoch {ep+1:>2}/{EPOCHS}  loss {tot/nb:.4f}  ({time.time()-t0:.0f}s)")

torch.save(model.state_dict(), "seq2seq.pt")
print("saved seq2seq.pt")

781 steps/epoch x 12 epochs = 9,372 steps
epoch  1/12  loss 0.7490  (235s)
epoch  2/12  loss 0.6942  (235s)
epoch  3/12  loss 0.6748  (236s)
epoch  4/12  loss 0.6658  (235s)
epoch  5/12  loss 0.6605  (235s)
epoch  6/12  loss 0.6576  (235s)
epoch  7/12  loss 0.6553  (235s)
epoch  8/12  loss 0.6500  (235s)
epoch  9/12  loss 0.6465  (236s)
epoch 10/12  loss 0.6434  (235s)
epoch 11/12  loss 0.6406  (235s)
epoch 12/12  loss 0.6391  (235s)
saved seq2seq.pt


## 7. Backtest

The evaluation reproduces the competition structure: fit through 2017-07-08,
skip a two-day gap, then forecast the following 62 days (2017-07-11 to
2017-09-10). The gap exists because the competition's forecast window opens
three days after the training data ends.

This is a walk-forward split. Suilin evaluated both this and a side-by-side
split and reports the latter to be uninformative for this problem: performance
on a held-out subset of series tracks training performance rather than future
performance.

The evaluation sample is stratified across traffic deciles. An unstratified
sample would be dominated by low-traffic articles — 18.7% of series average
under 10 views per day — and would obscure behaviour at the upper end, where an
earlier comparison of classical methods found the ranking of models to invert.

A paired bootstrap confidence interval accompanies the aggregate figure. This
is necessary rather than decorative: the observed advantage of 0.140 SMAPE has
an interval spanning zero, and the per-segment differences are larger than the
aggregate and of opposite sign in different segments. Reporting the aggregate
alone would misrepresent the result.

Scoring follows Kaggle's definition, in which SMAPE is 0 when actual and
forecast are both zero. Raw forecasts are retained so that per-series errors
remain available for significance testing.

In [12]:
def smape_cells(F, A):
    A = np.nan_to_num(A, nan=0.0); F = np.nan_to_num(np.maximum(F, 0), nan=0.0)
    den = np.abs(A) + np.abs(F)
    return np.where(den == 0, 0.0, 200 * np.abs(F - A) / np.where(den == 0, 1, den))


def stratified(n, seed=SEED):
    rng = np.random.default_rng(seed)
    edges = np.nanpercentile(med, np.linspace(0, 100, 11))
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        pool = np.where((med >= lo) & (med < hi))[0]
        if len(pool):
            out.append(rng.choice(pool, size=min(n // 10, len(pool)), replace=False))
    return np.sort(np.concatenate(out))


@torch.no_grad()
def predict(idx, origin, Varr=None, dowarr=None, bs=1024):
    model.eval(); out = []
    for k in range(0, len(idx), bs):
        b = [make_sample(int(i), origin, LOOKBACK, Varr, dowarr) for i in idx[k:k + bs]]
        t  = [torch.stack([x[j] for x in b]).to(DEVICE) for j in range(4)]
        ms = torch.stack([x[4] for x in b]).numpy()
        p  = model(*t).cpu().numpy()
        out.append(np.expm1(p * ms[:, 1:2] + ms[:, 0:1]))
    return np.vstack(out)


sel = stratified(2000)
P = predict(sel, TRN_END)
A = V[sel, -H:]
R = np.repeat(np.nan_to_num(np.nanmedian(V[sel, TRN_END-49:TRN_END], axis=1),
                            nan=0.0)[:, None], H, 1)

print(f"{'Median-49':<16}{smape_cells(np.round(R), A).mean():>8.3f}")
print(f"{'Seq2Seq+attn':<16}{smape_cells(np.round(P), A).mean():>8.3f}")

print("\nby traffic level")
for lo, hi, lbl in [(0,10,'<10'), (10,50,'10-50'), (50,200,'50-200'),
                    (200,1000,'200-1K'), (1000,1e12,'1K+')]:
    m = (med[sel] >= lo) & (med[sel] < hi)
    if m.sum():
        print(f"  {lbl:<8} n={m.sum():<5} median {smape_cells(np.round(R[m]), A[m]).mean():>6.2f}"
              f"   seq2seq {smape_cells(np.round(P[m]), A[m]).mean():>6.2f}")

# Retain raw forecasts. Per-series errors are required for significance testing
# and for regressing the model-baseline gap on series characteristics.
np.savez_compressed("seq2seq_backtest.npz", P=P, A=A, R=R, sel=sel, med_sel=med[sel])

# Paired bootstrap over series. Differences of this magnitude routinely fall
# within sampling error and must not be reported as improvements unqualified.
d_s2s = smape_cells(np.round(P), A).mean(axis=1)
d_med = smape_cells(np.round(R), A).mean(axis=1)
diff  = d_s2s - d_med                       # negative indicates the model wins
rng   = np.random.default_rng(0)
boot  = np.array([rng.choice(diff, len(diff), replace=True).mean() for _ in range(2000)])
lo_, hi_ = np.percentile(boot, [2.5, 97.5])
print(f"\nmean difference (seq2seq - median): {diff.mean():+.3f}")
print(f"95% bootstrap CI [{lo_:+.3f}, {hi_:+.3f}] -> "
      f"{'SIGNIFICANT' if (lo_ < 0) == (hi_ < 0) else 'not significant'}")
print(f"seq2seq wins on {(diff < 0).mean()*100:.1f}% of series")

Median-49         42.773
Seq2Seq+attn      42.633

by traffic level
  <10      n=385   median  66.47   seq2seq  68.80
  10-50    n=375   median  41.94   seq2seq  43.59
  50-200   n=385   median  39.92   seq2seq  38.47
  200-1K   n=565   median  35.34   seq2seq  33.47
  1K+      n=290   median  30.66   seq2seq  30.05

mean difference (seq2seq - median): -0.140
95% bootstrap CI [-1.154, +0.968] -> not significant
seq2seq wins on 46.2% of series


## 8. Submission

Forecasts are generated from the end of the observed data, covering 2017-09-13
to 2017-11-13. The encoder window is padded with missing values past the final
observation; at inference the target region is unused except to determine
tensor shape.

Predictions are mapped to submission identifiers by direct index lookup against
`key_2.csv` rather than by string join, avoiding an 8,993,906-row intermediate
frame. The assertions verify that every key resolves and the row count matches
exactly.

One limitation applies to any submission produced here. The model is trained on
data through 2017-07-08, because the backtest requires the subsequent period to
be held out. The two excluded months are the most recent, and therefore the most
informative for a forecast beginning 2017-09-13. A submission intended for
scoring should come from a model refit over the full history, changing
`Windows(SAMPLES, TRN_END)` to use the full length.

In [13]:
# Forecast from the end of the observed data. make_sample reads H days past the
# origin, so the array is padded; at inference the target is unused except to
# determine tensor shape.
FULL   = V.shape[1]
fut    = pd.date_range(dates[-1] + pd.Timedelta(days=3), periods=H)   # 2017-09-13..11-13
Vpad   = np.concatenate([V, np.full((V.shape[0], H), np.nan, "float32")], axis=1)
dowpad = np.concatenate([dow, np.array([d.dayofweek for d in fut], dtype="int64")])

P_sub = predict(np.arange(V.shape[0]), FULL, Vpad, dowpad)
del Vpad; gc.collect()
P_int = np.round(np.maximum(P_sub, 0)).astype("int32")
print("forecast matrix:", P_int.shape)

# Direct index lookup rather than a string join, which would materialise an
# 8,993,906-row intermediate frame.
assert len(np.unique(pages)) == len(pages), "Page values must be unique"
page_idx = pd.Series(np.arange(len(pages)), index=pages)
date_idx = pd.Series(np.arange(H), index=[d.strftime("%Y-%m-%d") for d in fut])

ids, vis, missing = [], [], 0
for chunk in pd.read_csv(os.path.join(DATA_DIR, "key_2.csv.zip"), chunksize=2_000_000):
    kp = chunk["Page"].str.rsplit("_", n=1, expand=True)
    r = kp[0].map(page_idx).to_numpy()
    c = kp[1].map(date_idx).to_numpy()
    ok = ~(pd.isna(r) | pd.isna(c))
    v = np.zeros(len(chunk), dtype="int64")
    v[ok] = P_int[r[ok].astype("int64"), c[ok].astype("int64")]
    missing += int((~ok).sum())
    ids.append(chunk["Id"].to_numpy()); vis.append(v)

sub = pd.DataFrame({"Id": np.concatenate(ids), "Visits": np.concatenate(vis)})
sub.to_csv("submission.csv", index=False)
print(f"submission.csv | {len(sub):,} rows | unmatched keys: {missing:,}")
assert missing == 0 and len(sub) == 8_993_906
sub.head()

forecast matrix: (145063, 62)
submission.csv | 8,993,906 rows | unmatched keys: 0


,Id,Visits
0,0b293039387a,430
1,7114389dd824,408
2,057b02ff1f09,418
3,bd2aca21caa3,482
4,c0effb42cdd5,518
